# PlanTo3D — the whole thing, on a GPU

Runs the app on Colab so the photoreal pass happens inside it rather than
as a separate step. Gradio hands back a public link you can open anywhere,
including on a phone.

**Set the runtime to GPU first:** Runtime → Change runtime type → T4 GPU.

Everything except the photoreal pass runs perfectly well on a laptop — the
segmenter takes seconds on a CPU. This exists because Stable Diffusion does
not, and running the app beside it is simpler than shuttling depth maps back
and forth by hand.

The link is temporary and lasts as long as the runtime.

## 1. Setup

In [ ]:
import torch

print(
    f"GPU: {torch.cuda.get_device_name(0)}"
    if torch.cuda.is_available()
    else "NO GPU — the app will still run, but without the photoreal pass."
)

In [ ]:
!apt-get -qq install -y poppler-utils tesseract-ocr > /dev/null
!pip install -q gradio trimesh shapely mapbox-earcut pytesseract pdf2image \
    segmentation-models-pytorch diffusers transformers accelerate safetensors
print("dependencies ready")

In [ ]:
import sys
from pathlib import Path

repo = Path("/content/PlanTo3D")
if repo.exists():
    !cd {repo} && git pull --quiet
else:
    !git clone --quiet https://github.com/priyanshsoni096-blip/PlanTo3D.git {repo}

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
%cd {repo}

## 2. The trained segmenter

Upload the checkpoint, or mount Drive if you keep it there. Without one the
app falls back to the classical baseline, which only reads clean CAD sheets.

In [ ]:
from google.colab import drive

models = Path("/content/PlanTo3D/models")
models.mkdir(exist_ok=True)

drive.mount("/content/drive")
checkpoint = Path("/content/drive/MyDrive/planto3d/unet_cubicasa.pt")

if checkpoint.is_file():
    !cp {checkpoint} {models}/unet_cubicasa.pt
    print(f"using {checkpoint}")
else:
    print("no checkpoint on Drive — upload one below, or the baseline is used")
    from google.colab import files

    for name in files.upload():
        Path(name).rename(models / name)

## 3. Load the diffusion pipeline

Loaded once, up front, so the first upload is not slowed by a 4 GB download.
Skip this cell to run the app without the photoreal pass.

In [ ]:
from diffusers import (
    ControlNetModel,
    StableDiffusionControlNetPipeline,
    UniPCMultistepScheduler,
)

PIPE = None
if torch.cuda.is_available():
    controlnet = ControlNetModel.from_pretrained(
        "lllyasviel/control_v11f1p_sd15_depth", torch_dtype=torch.float16
    )
    PIPE = StableDiffusionControlNetPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        controlnet=controlnet,
        torch_dtype=torch.float16,
        safety_checker=None,
    )
    PIPE.scheduler = UniPCMultistepScheduler.from_config(PIPE.scheduler.config)
    PIPE = PIPE.to("cuda")
    PIPE.enable_attention_slicing()
    print("photoreal pass ready")
else:
    print("no GPU — the app will run without the photoreal pass")

## 4. Launch

Same app as the desktop one, with the photoreal pass wired in. Open the
public link it prints — it works on a phone too.

**Upload the original PDF rather than screenshots.** Room names drive the
floor finishes, planting and railings, and reading them needs resolution.

In [ ]:
import gradio as gr

import app as planto3d_app
from planto3d.photoreal import NEGATIVE_PROMPT, build_guides, build_prompt


def photoreal(model_path, storeys, room_labels, strength, seed):
    """Render the model, then let diffusion dress it.

    The prompt is built from labels the pipeline actually read, so it
    describes this house rather than a generic one.
    """
    if PIPE is None:
        raise gr.Error(
            "The diffusion pipeline was not loaded. Run section 3, and check "
            "the runtime has a GPU: Runtime -> Change runtime type -> T4."
        )
    if not model_path:
        raise gr.Error("Convert a floor plan first — there is no model to render yet.")

    from PIL import Image

    guides = build_guides(model_path, Path(model_path).parent)
    depth = Image.open(guides["depth"]).convert("RGB")

    # Diffusion works in multiples of 8, and SD 1.5 is happiest near 768.
    longest = max(depth.size)
    if longest > 768:
        depth = depth.resize(tuple(int(d * 768 / longest) for d in depth.size))
    width, height = (max(d - d % 8, 8) for d in depth.size)
    print(f"generating at {width}x{height}, conditioning {strength}")

    return PIPE(
        prompt=build_prompt(storeys, room_labels),
        negative_prompt=NEGATIVE_PROMPT,
        image=depth.resize((width, height)),
        num_inference_steps=30,
        guidance_scale=8.0,
        controlnet_conditioning_scale=strength,
        generator=torch.Generator(device="cuda").manual_seed(int(seed)),
    ).images[0]


with gr.Blocks(title="PlanTo3D") as demo:
    gr.Markdown(f"# {planto3d_app.TITLE}\n{planto3d_app.DESCRIPTION}")

    with gr.Row():
        with gr.Column(scale=1):
            uploads = gr.File(
                label="Floor plan",
                file_types=[".pdf", ".png", ".jpg", ".jpeg"],
                file_count="multiple",
                type="filepath",
            )
            height_input = gr.Slider(7, 14, value=9, step=0.5, label="Storey height (feet)")
            convert_button = gr.Button("Convert to 3D", variant="primary")

            gr.Markdown("---\n### Photoreal pass")
            strength_input = gr.Slider(
                0.3,
                1.2,
                value=0.5,
                step=0.05,
                label="How tightly to hold the geometry",
                info="Lower gives richer materials and light; higher keeps the "
                "building closer to the plan.",
            )
            seed_input = gr.Number(value=7, label="Seed", precision=0)
            render_button = gr.Button("Make it photoreal")

        with gr.Column(scale=2):
            photo_output = gr.Image(label="Photoreal", height=420)
            hero_output = gr.Image(label="Rendered view", type="filepath", height=300)
            model_output = gr.Model3D(label="Interactive model", height=280)

    summary_output = gr.Markdown()
    view_output = gr.Gallery(label="Views", columns=3, height=320)
    overlay_output = gr.Gallery(label="What was detected", columns=3, height=300)

    # Carried between the two buttons, so the photoreal pass knows what the
    # conversion found rather than being told again.
    model_state = gr.State(None)
    labels_state = gr.State([])
    storeys_state = gr.State(3)

    def convert(files_in, storey_height):
        hero, model, views, overlays, summary, details = planto3d_app.convert_with_details(
            files_in, storey_height
        )
        return (
            hero, model, views, overlays, summary,
            model, details["labels"], details["storeys"],
        )

    convert_button.click(
        convert,
        [uploads, height_input],
        [
            hero_output,
            model_output,
            view_output,
            overlay_output,
            summary_output,
            model_state,
            labels_state,
            storeys_state,
        ],
    )
    render_button.click(
        photoreal,
        [model_state, storeys_state, labels_state, strength_input, seed_input],
        photo_output,
    )

demo.launch(share=True)